In [ ]:
!pip install unsloth

# Upgrade Unsloth from the latest repository
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.5/175.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 M

Found existing installation: unsloth 2025.1.8
Uninstalling unsloth-2025.1.8:
  Successfully uninstalled unsloth-2025.1.8
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-h4knmks0/unsloth_230cd247497949f2af2f19357dfbb43f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-h4knmks0/unsloth_230cd247497949f2af2f19357dfbb43f
  Resolved https://github.com/unslothai/unsloth.git to commit 038e6d4c8d40207a87297ab3aaf787c19b1006d1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.1.8-py3-none-any.whl size=174982 sha256=08d7afca36fbc5526e4be714e9b52344de1d3f57b260dcb7dbde0b7b1b3c927c
  Stored in directory: /tmp/pip-ephem-wheel-cache-zjrkwjkn/wheels/d1/17/05/850ab10c33284a4763b0595cd8ea9d01fce6e221cac24b3c01
Successfully built unsloth


In [ ]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")

login(token=hf_token, add_to_git_credential=True)

In [ ]:
import json
from collections import defaultdict
from datasets import Dataset
from transformers import AutoTokenizer


with open('/content/test.json', 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

#  Deduplicate courses & merge similar ones
course_dict = defaultdict(lambda: {"course_goals": set(), "course_results": set(), "course_program": set()})

for course in raw_data:
    course_name = course["course_name"]
    course_dict[course_name]["course_goals"].update(course["course_goals"].split(". "))
    course_dict[course_name]["course_results"].update(course["course_results"].split(". "))
    course_dict[course_name]["course_program"].update(course["course_program"].split(". "))

#  Convert sets back to formatted strings
for course_name, details in course_dict.items():
    details["course_goals"] = "\n- " + "\n- ".join(sorted(details["course_goals"]))
    details["course_results"] = "\n- " + "\n- ".join(sorted(details["course_results"]))
    details["course_program"] = "\n- " + "\n- ".join(sorted(details["course_program"]))

#  Load tokenizer
model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
EOS_TOKEN = tokenizer.eos_token  # End of sequence token


def format_training_data():
    structured_data = []
    for course_name, details in course_dict.items():
        prompt = f"""### Instrukcja:
Jesteś asystentem AI przeszkolonym do rekomendowania kursów edukacyjnych na podstawie poziomu studiów użytkownika oraz jego obszaru zainteresowań. Rekomenduj najlepszy kurs dla użytkownika, zapewniając, że:
- Kurs odpowiada poziomowi studiów użytkownika (Studia licencjackie/inżynierskie, Studia magisterskie, Jednolite studia magisterskie, Studia doktoranckie, Studia podyplomowe)
- Oferuje ustrukturyzowaną ścieżkę nauki
- Wyjaśnia kluczowe zagadnienia i oczekiwane rezultaty
- Nie powtarza danych wejściowych użytkownika


### Wejście:
Poziom studiów: {{study_level}}
Obszar studiów: {course_name}

### Odpowiedź:
Najlepszy kurs dla studenta na poziomie {{study_level}}, który studiuje w obszarze {course_name}, to:
"{course_name}"


### Cele kursu:
{details['course_goals']}

#### Oczekiwane rezultaty:
{details['course_results']}

#### Zakres tematów:
{details['course_program']}

"""
        structured_data.append({"text": prompt + EOS_TOKEN})

    return structured_data

#  Convert dataset to Hugging Face format
dataset = Dataset.from_list(format_training_data())


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [ ]:
dataset[0]

{'text': '### Instrukcja:\nJesteś asystentem AI przeszkolonym do rekomendowania kursów edukacyjnych na podstawie poziomu studiów użytkownika oraz jego obszaru zainteresowań. Rekomenduj najlepszy kurs dla użytkownika, zapewniając, że:\n- Kurs odpowiada poziomowi studiów użytkownika (Studia licencjackie/inżynierskie, Studia magisterskie, Jednolite studia magisterskie, Studia doktoranckie, Studia podyplomowe)\n- Oferuje ustrukturyzowaną ścieżkę nauki\n- Wyjaśnia kluczowe zagadnienia i oczekiwane rezultaty\n- Nie powtarza danych wejściowych użytkownika\n\n\n### Wejście:\nPoziom studiów: {study_level}\nObszar studiów: Instalator Pomp Ciepła\n\n### Odpowiedź:\nNajlepszy kurs dla studenta na poziomie {study_level}, który studiuje w obszarze Instalator Pomp Ciepła, to:\n"Instalator Pomp Ciepła"\n\n\n### Cele kursu:\n\n- Cel edukacyjny \n- Usługa prowadzi do nabycia wiedzy teoretycznej i praktycznej w zakresie doboru i montażu pomp ciepła.\n\n#### Oczekiwane rezultaty:\n\n- Uczestnik dysponuje 

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset


# Set parameters
max_seq_length = 2048
model_name = "meta-llama/Llama-3.2-3B"

# Load model with Unsloth (4-bit quantization)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# Apply PEFT (LoRA) for memory-efficient tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=32,
    loftq_config=None,
)

# Print trainable parameters
print(model.print_trainable_parameters())

# Training arguments
training_args = TrainingArguments(
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    warmup_steps=10,
    output_dir="llama-3.2-course-recommender",
    push_to_hub=True,
    seed=0,
    report_to="none",
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=True,
    args=training_args,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.1.8: Fast Llama patching. Transformers: 4.47.1.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth 2025.1.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 12,156,928 || all params: 3,224,906,752 || trainable%: 0.3770
None


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# Start training
trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 485 | Num Epochs = 5
O^O/ \_/ \    Batch size per device = 4 | Gradient Accumulation steps = 4
\        /    Total batch size = 16 | Total steps = 150
 "-____-"     Number of trainable parameters = 12,156,928


Step,Training Loss
1,1.541100


Step,Training Loss
1,1.541100
2,1.547700
3,1.555600
4,1.483300
5,1.520300
6,1.504000
7,1.402000
8,1.384600
9,1.345000
10,1.360700


TrainOutput(global_step=150, training_loss=0.9204245241483052, metrics={'train_runtime': 6534.6716, 'train_samples_per_second': 0.371, 'train_steps_per_second': 0.023, 'total_flos': 8.404334391853056e+16, 'train_loss': 0.9204245241483052, 'epoch': 4.983606557377049})

### Model Inference

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import pipeline, AutoTokenizer

model = FastLanguageModel.for_inference(model).to("cuda")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,  # Use float16 for low VRAM usage
)

print(" Model loaded successfully on GPU for inference!")

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausalLM', 'GraniteMoeForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianFor

 Model loaded successfully on GPU for inference!


In [20]:
def recommend_course(study_level, topic):
    prompt = f"""### Instrukcja:
Zarekomenduj kurs na poziomie {study_level} który interesuje się {topic}.

### Odpowiedź:
"""
    output = pipe(prompt, temperature=0.7, top_k=50)
    return output[0]['generated_text']

#Test the model
print(recommend_course("Studia licencjackie", "business inteligence"))

### Instrukcja:
Zarekomenduj kurs na poziomie Studia licencjackie który interesuje się business inteligence.

### Odpowiedź:
Najlepszy kurs dla studenta na poziomie {study_level}, który studiuje w obszarze business inteligence, to:
"Business inteligence"


### Cele kursu:

- Celem edukacyjnym szkolenia jest przygotowanie uczestników do efektywnego wykorzystania narzędzi AI w biznesie
- Szkolenie pozwoli uczestnikom zrozumieć podstawy AI, analizę danych, modelowanie decyzji oraz tworzenie inteligentnych aplikacji biznesowych.

#### Oczekiwane rezultaty:

- - zna podstawy AI, - zna metody analizy danych, - zna modelowanie decyzji, - zna tworzenie inteligentnych aplikacji biznesowych
- Uczestnik szkolenia nauczy się o programowaniu i automatyzacji procesów biznesowych, co przyczyni się do wzrostu efektywności i konkurencyjności firmy
- Uczestnik, który ukończył szkolenie : - zna metody analizy danych, - zna modelowanie decyzji, - zna tworzenie inteligentnych aplikacji biznesowych

#### Za